# 🚕 NYC Yellow Taxi Trip Data Ingestion

---

Welcome to the **NYC TAXI DATA PIPELINE - 2026**  
This pipeline ingests, audits, and loads the latest *Yellow Taxi* trips into the bronze layer.

---

> **Volume Source**  
> `/Volumes/nyctaxi/landing/operational/2026/YellowTaxi/`

---

- 🗂️ Data is loaded with **file name** and **load timestamp**  
- 💽 Written to: `NYCTAXI.BRONZE.YELLOW_TAXI`  
- 🟢 *Status*: *Ready for next processing step*

---

#### INGEST `TAXI TRIP FROM VOLUME`
- VOLUME = `/Volumes/nyctaxi/landing/operational/2026/`

In [0]:
from pyspark.sql.functions import input_file_name, col, current_timestamp

yellow_taxi_df = (spark.read.format('parquet')
                      .load('/Volumes/nyctaxi/landing/yellow_taxi/raw/2026/*')
                      .withColumn('file_name', col('_metadata.file_path'))
                      .withColumn('load_timestamp', current_timestamp())
                )
display(yellow_taxi_df.limit(1))

In [0]:
# LOAD THE RESULTANT DF INTO TEMP VIEW
yellow_taxi_df.createOrReplaceTempView('yellow_taxi_temp_vw')

In [0]:
%skip
%sql
CREATE OR REPLACE TEMP VIEW yellow_taxi_temp_vw_dedup AS
SELECT *
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, file_name
           ORDER BY load_time_stamp DESC
         ) AS rn
  FROM yellow_taxi_temp_vw
)
WHERE rn = 1

#### LOAD CURRENT YEAR YELLOW TAXI TRIP TO
- NYCTAXI.BRONZE.YELLOW_TAXI

In [0]:
%sql
SELECT MAX(load_timestamp) AS max_load_ts
        FROM NYCTAXI.BRONZE.YELLOW_TAXI
        WHERE file_name not like '%2025%';

MERGE INTO NYCTAXI.BRONZE.YELLOW_TAXI tgt
USING (SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, file_name ORDER BY load_timestamp DESC) as rn FROM yellow_taxi_temp_vw) WHERE rn = 1) src
ON tgt.VendorID = src.VendorID
   AND tgt.tpep_pickup_datetime = src.tpep_pickup_datetime
   AND tgt.tpep_dropoff_datetime = src.tpep_dropoff_datetime
   AND tgt.file_name = src.file_name

WHEN MATCHED THEN
  UPDATE SET
    passenger_count = src.passenger_count,
    trip_distance = src.trip_distance,
    RatecodeID = src.RatecodeID,
    store_and_fwd_flag = src.store_and_fwd_flag,
    PULocationID = src.PULocationID,
    DOLocationID = src.DOLocationID,
    payment_type = src.payment_type,
    fare_amount = src.fare_amount,
    extra = src.extra,
    mta_tax = src.mta_tax,
    tip_amount = src.tip_amount,
    tolls_amount = src.tolls_amount,
    improvement_surcharge = src.improvement_surcharge,
    total_amount = src.total_amount,
    congestion_surcharge = src.congestion_surcharge,
    Airport_fee = src.Airport_fee,
    cbd_congestion_fee = src.cbd_congestion_fee,
    file_name = src.file_name,
    load_timestamp = src.load_timestamp

WHEN NOT MATCHED THEN
  INSERT (
    VendorID,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    passenger_count,
    trip_distance,
    RatecodeID,
    store_and_fwd_flag,
    PULocationID,
    DOLocationID,
    payment_type,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    improvement_surcharge,
    total_amount,
    congestion_surcharge,
    Airport_fee,
    cbd_congestion_fee,
    file_name,
    load_timestamp
  )
  VALUES (
    src.VendorID,
    src.tpep_pickup_datetime,
    src.tpep_dropoff_datetime,
    src.passenger_count,
    src.trip_distance,
    src.RatecodeID,
    src.store_and_fwd_flag,
    src.PULocationID,
    src.DOLocationID,
    src.payment_type,
    src.fare_amount,
    src.extra,
    src.mta_tax,
    src.tip_amount,
    src.tolls_amount,
    src.improvement_surcharge,
    src.total_amount,
    src.congestion_surcharge,
    src.Airport_fee,
    src.cbd_congestion_fee,
    src.file_name,
    src.load_timestamp
  );

In [0]:
dbutils.notebook.exit('YELLOW TAXI TRIP HAS BEEN LOADED INTO NYCTAXI.BRONZE.YELLOW_TAXI')